# Model Training Notebook
This notebook provides a complete workflow for data mining and machine learning model training.

## 1. Import Libraries

In [1]:
import sys
print(sys.executable)

c:\Users\ADMIN\AppData\Local\Programs\Python\Python311\python.exe


In [2]:
from sklearn.svm import LinearSVC
import numpy as np
import pandas as pd

In [3]:
data = pd.read_csv('CSV/train_papers_topics_keywords.csv')  
data_test = pd.concat([pd.read_csv('CSV/public_test_topics_keywords.csv'), pd.read_csv('CSV/private_test_topics_keywords.csv')])
#data_test=pd.read_csv('CSV/public_test_with_abstracts_updated_4_2.csv')
#data_test=pd.read_csv('CSV/private_test_with_abstracts_update.csv')
length = len(data_test)
print(length)

596


In [4]:
#Find the data types of each column
data.dtypes

id                       int64
title                   object
venue                   object
year                     int64
authors                 object
doi                     object
Label                    int64
resolved_doi            object
openalex_id             object
semantic_scholar_id     object
primary_topic           object
primary_topic_score    float64
topic_1                 object
topic_1_score          float64
topic_2                 object
topic_2_score          float64
topic_3                 object
topic_3_score          float64
keyword_1               object
keyword_1_score        float64
keyword_2               object
keyword_2_score        float64
keyword_3               object
keyword_3_score        float64
topics_json             object
keywords_json           object
fetch_source            object
dtype: object

## 2. Clean Data 


In [5]:
#Check for missing values
#data.isnull().sum()
data_test.isnull().sum()

id                       0
title                    0
venue                    0
year                     0
authors                 41
doi                      0
resolved_doi            61
openalex_id             20
semantic_scholar_id    361
primary_topic            3
primary_topic_score      3
topic_1                  3
topic_1_score            3
topic_2                 25
topic_2_score           25
topic_3                 48
topic_3_score           48
keyword_1                3
keyword_1_score          3
keyword_2               13
keyword_2_score         13
keyword_3               20
keyword_3_score         20
topics_json              2
keywords_json            2
fetch_source             2
dtype: int64

In [28]:
#Replace missing values of the authors column with unknown
columns_to_fill = ['authors']  # Specify which columns to fill
for col in columns_to_fill:
    if col in data.columns:
        data[col] = data[col].fillna('unknown')
columns_to_fill = ['authors']  # Specify which columns to fill
for col in columns_to_fill:
    if col in data_test.columns:
        data_test[col] = data_test[col].fillna('unknown')

In [29]:
#Recheck for missing values
data_test.isnull().sum()

id                   0
title                0
venue                0
year                 0
authors              0
doi                  0
abstract            96
abstract_source    132
dtype: int64

In [6]:
data.isnull().sum()

id                        0
title                     0
venue                     0
year                      0
authors                 204
doi                       0
Label                     0
resolved_doi            279
openalex_id             648
semantic_scholar_id    1363
primary_topic             2
primary_topic_score       2
topic_1                   2
topic_1_score             2
topic_2                 358
topic_2_score           358
topic_3                 739
topic_3_score           739
keyword_1                 0
keyword_1_score           0
keyword_2                24
keyword_2_score          24
keyword_3                41
keyword_3_score          41
topics_json               0
keywords_json             0
fetch_source              0
dtype: int64

In [31]:
#Check if there is non ASCII text in the title and abstract columns
column_to_check = ['title']  # Specify which columns to check
for col in column_to_check:
    if col in data.columns:
        non_ASCII = data[~data[col].apply(lambda x: isinstance(x, str) and all(ord(c) < 128 for c in x))]
        print(f"Non-ASCII entries in column '{col}':")

Non-ASCII entries in column 'title':


## 3. TRAIN THE MODEL


In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.pipeline import Pipeline
import numpy as np

# =========================
# Load datasets
# =========================

# Data already loaded in previous cells
# data = train_cleaned_phong.csv
# data_test = concatenation of public_test and private_test with abstracts

# =========================
# Clean and prepare data - Remove rows with missing abstracts
# =========================

# Check for missing abstracts in training data
print("Before filtering:")
print(f"Training data shape: {data.shape}")
print(f"Missing abstracts in training data: {data['abstract'].isna().sum()}")

# Remove rows where abstract is NaN or empty
data_filtered = data[data['abstract'].notna() & (data['abstract'].str.strip() != '')].copy()

print(f"\nAfter filtering:")
print(f"Training data shape: {data_filtered.shape}")
print(f"Removed {len(data) - len(data_filtered)} rows with missing/empty abstracts")

# Prepare training labels
y_train = data_filtered["Label"]

# =========================
# Prepare test data
# =========================

X_test_ids = data_test['id'].copy()
X_test_abstract = data_test["abstract"].astype(str)
X_test_title = data_test["title"].astype(str)

# =========================
# METHOD 1: Using ABSTRACT ONLY
# =========================

print("\n" + "="*50)
print("METHOD 1: ABSTRACT ONLY")
print("="*50)

X_train_abstract = data_filtered["abstract"].astype(str)

# --- Model 1a: Logistic Regression with Abstract ---
print("\n1a. Training Logistic Regression (Abstract Only)...")
model_1a = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("logistic_regression", LogisticRegression(max_iter=1000, random_state=42))
])

model_1a.fit(X_train_abstract, y_train)
predictions_1a = model_1a.predict(X_test_abstract)
predictions_1a = predictions_1a.astype(int)
print(f"✓ Logistic Regression (Abstract) - Predictions shape: {predictions_1a.shape}")

# --- Model 1b: Linear Regression with Abstract ---
print("\n1b. Training Linear Regression (Abstract Only)...")
model_1b = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("linear_regression", LinearRegression())
])

model_1b.fit(X_train_abstract, y_train)
predictions_1b = model_1b.predict(X_test_abstract)
predictions_1b = np.round(predictions_1b).clip(1, 5).astype(int)
print(f"✓ Linear Regression (Abstract) - Predictions shape: {predictions_1b.shape}")

# =========================
# METHOD 2: Using TITLE + ABSTRACT
# =========================

print("\n" + "="*50)
print("METHOD 2: TITLE + ABSTRACT (COMBINED)")
print("="*50)

X_train_title = data_filtered["title"].astype(str)
X_train_combined = X_train_title + " " + X_train_abstract

X_test_combined = X_test_title + " " + X_test_abstract

# --- Model 2a: Logistic Regression with Title + Abstract ---
print("\n2a. Training Logistic Regression (Title + Abstract)...")
model_2a = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("logistic_regression", LogisticRegression(max_iter=1000, random_state=42))
])

model_2a.fit(X_train_combined, y_train)
predictions_2a = model_2a.predict(X_test_combined)
predictions_2a = predictions_2a.astype(int)
print(f"✓ Logistic Regression (Title + Abstract) - Predictions shape: {predictions_2a.shape}")

# --- Model 2b: Linear Regression with Title + Abstract ---
print("\n2b. Training Linear Regression (Title + Abstract)...")
model_2b = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000)),
    ("linear_regression", LinearRegression())
])

model_2b.fit(X_train_combined, y_train)
predictions_2b = model_2b.predict(X_test_combined)
predictions_2b = np.round(predictions_2b).clip(1, 5).astype(int)
print(f"✓ Linear Regression (Title + Abstract) - Predictions shape: {predictions_2b.shape}")

# =========================
# Save predictions
# =========================

print("\n" + "="*50)
print("SAVING PREDICTIONS")
print("="*50)

# Save predictions from all 4 models
results_all_models = pd.DataFrame({
    'id': X_test_ids,
    'Abstract_LogisticRegression': predictions_1a,
    'Abstract_LinearRegression': predictions_1b,
    'TitleAbstract_LogisticRegression': predictions_2a,
    'TitleAbstract_LinearRegression': predictions_2b
})

# Use the first model's predictions as default for submission (Title + Abstract with Logistic Regression)
predictions_default = predictions_1a

results_df = pd.DataFrame({
    'id': X_test_ids,
    'Label': predictions_default
})

results_df.to_csv('predictions.csv', index=False)
results_all_models.to_csv('predictions_all_models.csv', index=False)

print(f"\n✓ predictions.csv saved ({len(results_df)} rows)")
print(f"✓ predictions_all_models.csv saved (contains all 4 model predictions)")

# Display sample predictions
print("\n" + "="*50)
print("SAMPLE PREDICTIONS (First 10 rows)")
print("="*50)
print(results_all_models.head(10))

Before filtering:
Training data shape: (1933, 9)
Missing abstracts in training data: 0

After filtering:
Training data shape: (1933, 9)
Removed 0 rows with missing/empty abstracts

METHOD 1: ABSTRACT ONLY

1a. Training Logistic Regression (Abstract Only)...
✓ Logistic Regression (Abstract) - Predictions shape: (596,)

1b. Training Linear Regression (Abstract Only)...
✓ Linear Regression (Abstract) - Predictions shape: (596,)

METHOD 2: TITLE + ABSTRACT (COMBINED)

2a. Training Logistic Regression (Title + Abstract)...
✓ Logistic Regression (Title + Abstract) - Predictions shape: (596,)

2b. Training Linear Regression (Title + Abstract)...
✓ Linear Regression (Title + Abstract) - Predictions shape: (596,)

SAVING PREDICTIONS

✓ predictions.csv saved (596 rows)
✓ predictions_all_models.csv saved (contains all 4 model predictions)

SAMPLE PREDICTIONS (First 10 rows)
     id  Abstract_LogisticRegression  Abstract_LinearRegression  \
0   979                            1                     